# Imports

In [0]:
from pyspark.sql import functions as F
from datetime import timedelta
from pyspark.sql import Window as W
from pyspark.sql.types import DecimalType, IntegerType, TimestampType, DateType, DoubleType

spark.sql("CREATE DATABASE IF NOT EXISTS silver")

def rename_cols(df, mapping: dict):
    for old, new in mapping.items():
        if old in df.columns:
            df = df.withColumnRenamed(old, new)
    return df

# ft_consumidores para silver

In [0]:
df_bz = spark.table("bronze.ft_consumidores")

mapping = {
    "customer_id": "id_consumidor",
    "customer_zip_code_prefix": "prefixo_cep",
    "customer_city": "cidade",
    "customer_state": "estado",
}
df = rename_cols(df_bz, mapping)

# Tipagem
df = (df
      .withColumn("prefixo_cep", F.col("prefixo_cep").cast(IntegerType()))
      .withColumn("cidade", F.upper(F.col("cidade")))
      .withColumn("estado", F.upper(F.col("estado")))
     )

# Deduplicação por id_consumidor mantendo o mais recente pela ingestion_timestamp
win = W.partitionBy("id_consumidor").orderBy(F.col("ingestion_timestamp").desc())
df = (df
      .withColumn("rn", F.row_number().over(win))
      .filter(F.col("rn") == 1)
      .drop("rn"))

# Salvar
df.write.mode("overwrite").saveAsTable("silver.ft_consumidores")
print(f"silver.ft_consumidores: {df.count()} rows")

display(df.limit(10))

# t_pedidos para silver

In [0]:
df_bz = spark.table("bronze.ft_pedidos")

mapping = {
    "order_id": "id_pedido",
    "customer_id": "id_consumidor",
    "order_status": "status",
    "order_purchase_timestamp": "pedido_compra_timestamp",
    "order_approved_at": "pedido_aprovado_timestamp",
    "order_delivered_carrier_date": "pedido_carregado_timestamp",
    "order_delivered_customer_date": "pedido_entregue_timestamp",
    "order_estimated_delivery_date": "pedido_estimativa_entrega_timestamp",
}
df = rename_cols(df_bz, mapping)

# Tipagem
ts_cols = ["pedido_compra_timestamp","pedido_aprovado_timestamp",
           "pedido_carregado_timestamp","pedido_entregue_timestamp",
           "pedido_estimativa_entrega_timestamp"]
for c in ts_cols:
    df = df.withColumn(c, F.col(c).cast(TimestampType()))

# Tradução de status
status_map = {
    "delivered": "entregue",
    "invoiced": "faturado",
    "shipped": "enviado",
    "processing": "em processamento",
    "unavailable": "indisponível",
    "canceled": "cancelado",
    "created": "criado",
    "approved": "aprovado",
}
mapping_expr = F.create_map([F.lit(x) for kv in status_map.items() for x in kv])
df = df.withColumn("status", F.coalesce(mapping_expr[F.col("status")], F.col("status")))

# Derivadas
df = df.withColumn(
    "tempo_entrega_dias",
    F.when(F.col("pedido_entregue_timestamp").isNotNull(),
           F.datediff(F.col("pedido_entregue_timestamp"), F.col("pedido_compra_timestamp")))
)
df = df.withColumn(
    "tempo_entrega_estimado_dias",
    F.when(F.col("pedido_estimativa_entrega_timestamp").isNotNull(),
           F.datediff(F.col("pedido_estimativa_entrega_timestamp"), F.col("pedido_compra_timestamp")))
)
df = df.withColumn(
    "diferenca_entrega_dias",
    F.when(F.col("tempo_entrega_dias").isNotNull() & F.col("tempo_entrega_estimado_dias").isNotNull(),
           F.col("tempo_entrega_dias") - F.col("tempo_entrega_estimado_dias"))
)
df = df.withColumn(
    "entrega_no_prazo",
    F.when(F.col("pedido_entregue_timestamp").isNull(), F.lit("Não Entregue"))
     .when(F.col("diferenca_entrega_dias") <= 0, F.lit("Sim"))
     .otherwise(F.lit("Não"))
)

df.write.mode("overwrite").saveAsTable("silver.ft_pedidos")
print(f"silver.ft_pedidos: {df.count()} rows")

display(df.limit(10))

# ft_itens_pedidos para silver

In [0]:
df_bz = spark.table("bronze.ft_itens_pedidos")

mapping = {
    "order_id": "id_pedido",
    "order_item_id": "id_item",
    "product_id": "id_produto",
    "seller_id": "id_vendedor",
    "price": "preco_brl",
    "freight_value": "preco_frete",
}
df = rename_cols(df_bz, mapping)

df = (df
      .withColumn("id_item", F.col("id_item").cast(IntegerType()))
      .withColumn("preco_brl", F.col("preco_brl").cast(DecimalType(12,2)))
      .withColumn("preco_frete", F.col("preco_frete").cast(DecimalType(12,2)))
     )

df.write.mode("overwrite").saveAsTable("silver.ft_itens_pedidos")
print(f"silver.ft_itens_pedidos: {df.count()} rows")

display(df.limit(10))


# ft_pagamentos_pedidos para silver

In [0]:
df_bz = spark.table("bronze.ft_pagamentos_pedidos")

mapping = {
    "order_id": "id_pedido",
    "payment_sequential": "codigo_pagamento",
    "payment_type": "forma_pagamento",
    "payment_installments": "parcelas",
    "payment_value": "valor_pagamento",
}
df = rename_cols(df_bz, mapping)

pay_map = {
    "credit_card": "Cartão de Crédito",
    "boleto": "Boleto",
    "voucher": "Voucher",
    "debit_card": "Cartão de Débito",
}
mapping_expr = F.create_map([F.lit(x) for kv in pay_map.items() for x in kv])
df = df.withColumn("forma_pagamento", F.coalesce(mapping_expr[F.col("forma_pagamento")], F.lit("Outro")))

df = (df
      .withColumn("codigo_pagamento", F.col("codigo_pagamento").cast(IntegerType()))
      .withColumn("parcelas", F.col("parcelas").cast(IntegerType()))
      .withColumn("valor_pagamento", F.col("valor_pagamento").cast(DecimalType(12,2)))
     )

df.write.mode("overwrite").saveAsTable("silver.ft_pagamentos_pedidos")
print(f"silver.ft_pagamentos_pedidos: {df.count()} rows")

display(df.limit(10))

# ft_avaliacoes_pedidos silver

“ID incorreto”:
Consideramos ID incorreto quando:

id_pedido é nulo ou id_pedido não existe em bronze.ft_pedidos (checado por anti-join contra order_id, com referência id_pedido_ref).
Essas linhas são descartadas ao final (apenas inner join com a referência é mantido).

“Data preenchida errada”:
Consideramos data preenchida errada quando ocorre qualquer uma das condições:

data_comentario é nula (inclui textos que não puderam ser parseados);

data_comentario está no futuro em relação à data atual (current_date());

data_resposta existe e sua data (to_date(data_resposta)) está no futuro.
Linhas que violam qualquer uma dessas regras são removidas.

In [0]:
df_bz = spark.table("bronze.ft_avaliacoes_pedidos")
pedidos_ref = (spark.table("bronze.ft_pedidos")
               .select("order_id").withColumnRenamed("order_id", "id_pedido_ref").distinct())

mapping = {
    "review_id": "id_avaliacao",
    "order_id": "id_pedido",
    "review_score": "avaliacao",
    "review_comment_title": "titulo_comentario",
    "review_comment_message": "comentario",
    "review_creation_date": "data_comentario",
    "review_answer_timestamp": "data_resposta",
}
df = rename_cols(df_bz, mapping)

df = (df
      .withColumn("data_comentario_str", F.col("data_comentario").cast("string"))
      .withColumn("data_resposta_str",  F.col("data_resposta").cast("string"))
     )

df = df.withColumn(
    "data_comentario_ts",
    F.coalesce(
        F.expr("try_to_timestamp(data_comentario_str, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(data_comentario_str, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(data_comentario_str, 'dd/MM/yyyy HH:mm:ss')"),
        F.expr("try_to_timestamp(data_comentario_str, 'dd/MM/yyyy')"),
        F.expr("try_cast(data_comentario_str as timestamp)")
    )
).withColumn(
    "data_comentario",
    F.to_date("data_comentario_ts").cast(DateType())
).drop("data_comentario_ts")

df = df.withColumn(
    "data_resposta",
    F.coalesce(
        F.expr("try_to_timestamp(data_resposta_str, 'yyyy-MM-dd HH:mm:ss')"),
        F.expr("try_to_timestamp(data_resposta_str, 'yyyy-MM-dd')"),
        F.expr("try_to_timestamp(data_resposta_str, 'dd/MM/yyyy HH:mm:ss')"),
        F.expr("try_to_timestamp(data_resposta_str, 'dd/MM/yyyy')"),
        F.expr("try_cast(data_resposta_str as timestamp)")
    ).cast(TimestampType())
).drop("data_resposta_str")

# Regras de qualidade
today = F.current_date()
cond_bad_date = (
    F.col("data_comentario").isNull() |
    (F.col("data_comentario") > today) |
    (F.col("data_resposta").isNotNull() & (F.to_date("data_resposta") > today))
)

# Contagens (para documentar no caderno)
cnt_total_in = df.count()
cnt_null_id  = df.filter(F.col("id_pedido").isNull()).count()
cnt_not_found = (
    df.filter(F.col("id_pedido").isNotNull())
      .join(pedidos_ref, df.id_pedido == pedidos_ref.id_pedido_ref, "left_anti")
      .count()
)
cnt_bad_date = df.filter(cond_bad_date).count()

print(f"[INPUT] total rows: {cnt_total_in}")
print(f"[RULE] invalid id_pedido (NULL): {cnt_null_id}")
print(f"[RULE] invalid id_pedido (not found in bronze.ft_pedidos): {cnt_not_found}")
print(f"[RULE] bad dates (null/future): {cnt_bad_date}")

# Remoção efetiva
df_valid = (
    df.join(pedidos_ref, df.id_pedido == pedidos_ref.id_pedido_ref, "inner")
      .filter(~cond_bad_date)
      .drop("id_pedido_ref")
)

cnt_out = df_valid.count()
print(f"[OUTPUT] kept rows: {cnt_out} | removed rows: {cnt_total_in - cnt_out}")

df_valid.write.mode("overwrite").saveAsTable("silver.ft_avaliacoes_pedidos")
display(df_valid.limit(10))

# ft_produtos para silver

In [0]:
df_bz = spark.table("bronze.ft_produtos")

mapping = {
    "product_id": "id_produto",
    "product_category_name": "categoria_produto",
    "product_weight_g": "peso_produto_gramas",
    "product_length_cm": "comprimento_centimetros",
    "product_height_cm": "altura_centimetros",
    "product_width_cm": "largura_centimetros",
}
df = rename_cols(df_bz, mapping)

int_cols = ["peso_produto_gramas","comprimento_centimetros","altura_centimetros","largura_centimetros"]
for c in int_cols:
    if c in df.columns:
        df = df.withColumn(c, F.col(c).cast(IntegerType()))

df.write.mode("overwrite").saveAsTable("silver.ft_produtos")
print(f"silver.ft_produtos: {df.count()} rows")

display(df.limit(10))

# ft_vendedores para silver

In [0]:
df_bz = spark.table("bronze.ft_vendedores")

mapping = {
    "seller_id": "id_vendedor",
    "seller_zip_code_prefix": "prefixo_cep",
    "seller_city": "cidade",
    "seller_state": "estado",
}
df = rename_cols(df_bz, mapping)

df = (df
      .withColumn("prefixo_cep", F.col("prefixo_cep").cast(IntegerType()))
      .withColumn("cidade", F.upper(F.col("cidade")))
      .withColumn("estado", F.upper(F.col("estado")))
     )

df.write.mode("overwrite").saveAsTable("silver.ft_vendedores")
print(f"silver.ft_vendedores: {df.count()} rows")

display(df.limit(10))

# dm_categoria_produtos_traducao para silver

In [0]:
df_bz = spark.table("bronze.dm_categoria_produtos_traducao")

mapping = {
    "product_category_name": "nome_produto_pt",
    "product_category_name_english": "nome_produto_en",
}
df = rename_cols(df_bz, mapping)

df.write.mode("overwrite").saveAsTable("silver.dm_categoria_produtos_traducao")
print(f"silver.dm_categoria_produtos_traducao: {df.count()} rows")

display(df.limit(10))

# dm_cotacao_dolar para silver

In [0]:
# Descobrir o intervalo de datas
minmax = (spark.table("bronze.dm_cotacao_dolar")
          .select(F.to_date(F.to_timestamp("dataHoraCotacao")).alias("d"))
          .agg(F.min("d").alias("min_d"), F.max("d").alias("max_d"))
          .first())
min_d, max_d = minmax.min_d, minmax.max_d

# Buffer para o primeiro fim de semana seja preenchido com a sexta anterior
start_with_buffer = min_d - timedelta(days=7)

# Construir um calendário diário (inclusivo)
date_span = (spark.range(1)
             .select(F.explode(F.sequence(F.lit(start_with_buffer), F.lit(max_d))).alias("data")))

# Normalizar as cotações da Bronze e escolher o fechamento
rates_raw = (spark.table("bronze.dm_cotacao_dolar")
             .withColumn("ts", F.to_timestamp("dataHoraCotacao"))
             .withColumn("data", F.to_date("ts"))
             .withColumn("cotacao_dolar", F.col("cotacaoCompra").cast(DoubleType()))
             .select("data", "ts", "cotacao_dolar")
             )

# Para cada data, pegar a última cotação do dia (fechamento)
w_close = W.partitionBy("data").orderBy(F.col("ts").desc())
rates_daily = (rates_raw
               .withColumn("rn", F.row_number().over(w_close))
               .filter(F.col("rn") == 1)
               .select("data", "cotacao_dolar"))

# Forward-fill sobre o calendário (fins de semana recebem a cotação da sexta)
w_ffill = W.partitionBy(F.lit(1)).orderBy("data").rowsBetween(W.unboundedPreceding, 0)
rates_full = (date_span.join(rates_daily, "data", "left")
              .withColumn("cotacao_dolar", F.last("cotacao_dolar", ignorenulls=True).over(w_ffill)))

# Manter apenas datas >= min_d e selecionar o schema final
df_silver = (rates_full
    .filter(F.col("data") >= F.lit(min_d))
    .select(
        F.round(F.col("cotacao_dolar"), 2).cast(DecimalType(12, 2)).alias("cotacao_dolar"),
        F.col("data").cast(DateType()).alias("data")
    )
)

# Salvar na Silver
spark.sql("CREATE DATABASE IF NOT EXISTS silver")
df_silver.write.mode("overwrite").saveAsTable("silver.dm_cotacao_dolar")

# Checagens
print("Rows:", df_silver.count(),
      "| NULL cotacao_dolar:", df_silver.filter(F.col("cotacao_dolar").isNull()).count(),
      "| range:", df_silver.agg(F.min("data"), F.max("data")).first())

display(df_silver.orderBy("data").limit(10))

# Verificações de integridade referencial

In [0]:
pedidos = spark.table("silver.ft_pedidos")
consumidores = spark.table("silver.ft_consumidores")
itens = spark.table("silver.ft_itens_pedidos")

# Pedidos órfãos (sem consumidor)
pedidos_orfaos = pedidos.join(consumidores.select("id_consumidor").distinct(),
                              "id_consumidor", "left_anti")
n_pedidos_orfaos = pedidos_orfaos.count()
print(f"Pedidos órfãos: {n_pedidos_orfaos}")

pedidos_ok = pedidos.join(consumidores.select("id_consumidor").distinct(),
                          "id_consumidor", "inner")

# Itens órfãos (sem pedido)
itens_orfaos = itens.join(pedidos_ok.select("id_pedido").distinct(),
                          "id_pedido", "left_anti")
n_itens_orfaos = itens_orfaos.count()
print(f"Itens órfãos: {n_itens_orfaos}")

itens_ok = itens.join(pedidos_ok.select("id_pedido").distinct(), "id_pedido", "inner")

# Persistir versões sem órfãos
pedidos_ok.write.mode("overwrite").saveAsTable("silver.ft_pedidos")
itens_ok.write.mode("overwrite").saveAsTable("silver.ft_itens_pedidos")

# silver.ft_pedido_total

In [0]:
# Bases
orders = (spark.table("silver.ft_pedidos")
          .select("id_pedido", "id_consumidor", "status", "pedido_compra_timestamp")
          .withColumn("data_pedido", F.to_date("pedido_compra_timestamp")))

payments = spark.table("silver.ft_pagamentos_pedidos").select("id_pedido", "valor_pagamento")
usd = spark.table("silver.dm_cotacao_dolar").select(
    F.col("data").alias("data_pedido"),
    F.col("cotacao_dolar")
)

# Pagamentos agregados por pedido (BRL)
pay_agg = (payments
           .groupBy("id_pedido")
           .agg(F.sum("valor_pagamento").alias("valor_total_pago_brl")))

# Join pedidos + pagamentos + cotação do dia
ft = (orders
      .join(pay_agg, "id_pedido", "left")
      .join(usd, "data_pedido", "left"))

# Tipagem e arredondamento
ft = (ft
      .withColumn("valor_total_pago_brl",
                  F.round(F.col("valor_total_pago_brl"), 2).cast(DecimalType(12, 2)))
      .withColumn("valor_total_pago_usd",
                  F.when(F.col("valor_total_pago_brl").isNotNull() & F.col("cotacao_dolar").isNotNull(),
                         F.round(
                             (F.col("valor_total_pago_brl").cast(DoubleType()) /
                              F.col("cotacao_dolar").cast(DoubleType())), 2
                         )
                  ).cast(DecimalType(12, 2)))
     )

# Seleção final
ft_final = ft.select(
    "id_pedido",
    "id_consumidor",
    "status",
    "valor_total_pago_brl",
    "valor_total_pago_usd",
    "data_pedido"
)

ft_final.write.mode("overwrite").saveAsTable("silver.ft_pedido_total")

print(f"silver.ft_pedido_total created with {ft_final.count()} rows")
display(ft_final.orderBy("data_pedido").limit(10))